# Quickstart: loading the clean NERSC data

This notebook shows how to load and explore the clean CSVs via `pipeline.io`.

**Before running this**: from the repo root, run `python -m pipeline.run` once to generate `clean/`.


In [ ]:
import sys; sys.path.append('..')
from pipeline import io as cleaned_io

sorted(p.name for p in cleaned_io.CLEANED_DIR.glob('*.csv'))


## 1. Load one year of allocations

In [ ]:
alloc_2024 = cleaned_io.read_allocations(2024)
print(alloc_2024.shape)
alloc_2024.head()


## 2. Load all years at once

In [ ]:
alloc_all = cleaned_io.read_all_allocations()
alloc_all.groupby('year').size()


## 3a. Users summary tables

In [ ]:
users = cleaned_io.read_users()
print('users:', users.shape)
users.head()


## 3b. Repos summary tables

In [ ]:
repos = cleaned_io.read_repos()
print('repos:', repos.shape)
repos.head()


## 4. Example explorations

### Top 10 repos by lifetime user count

In [ ]:
repos.sort_values('n_users_lifetime', ascending=False).head(10)

### Users active in the most years

In [ ]:
users.sort_values('tenure', ascending=False).head(10)


### GPU vs CPU node-hours per year, across all repos

In [ ]:
alloc_all.groupby('year')[['cpu_node_hours_charged', 'gpu_node_hours_charged']].sum()

### Pick the repo that appears in the most years, then list its project-lead handle per year.

In [ ]:
repo_years = alloc_all.groupby('repo')['year'].nunique().sort_values(ascending=False)
long_repo = repo_years.index[0]
print('Longest-standing repo:', long_repo, '(', repo_years.iloc[0], 'years )')

pi_by_year = (alloc_all[alloc_all['repo'] == long_repo]
              .groupby('year')['pi_user_id'].unique())
pi_by_year

### The set of all project-lead handles across years.

In [ ]:
all_pis = set(alloc_all[alloc_all['repo'] == long_repo]['pi_user_id'].dropna())
print('Distinct project-lead handles for', long_repo, ':', all_pis)